In [ ]:
import numpy as np
import json
import pandas as pd
from datetime import timedelta
import random
import pycountry
from numpyro import distributions as dist
import arviz as az
from IPython.display import display, Markdown

from emu_renewal.constants import DATA_PATH, RERUNS, FULL_RUN, TIMEOUTS, DATE_FORMAT, RUN_DATA_DELAY, EXP_PRIOR_LOWER, EXP_PRIOR_UPPER
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.priors import get_standard_priors
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget
from emu_renewal.plotting import plot_input_recovery
from emu_renewal.utils import get_analysis_paths

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
sample_countries = random.sample([c for c in analysis_paths if "fb_visited_mob" in analysis_paths[c]], 4)

In [ ]:
for iso3 in sample_countries[1:]:
    mob_source = random.sample(["fb_visited_mob", "fb_singletile_mob"], 1)[0]
    name = pycountry.countries.lookup(iso3).name
    continent = get_cont_of_country(iso3)

    display(Markdown(f"mobility approach: {mob_source}"))
    display(Markdown(f"\n country: {name}"))
    
    # Select random set of scalar parameters from calibration posterior
    path = analysis_paths[iso3][mob_source]
    idata = az.from_netcdf(path / "idata_filtered.nc")
    post = idata.posterior
    chain = np.random.randint(post.sizes["chain"])
    draw = np.random.randint(post.sizes["draw"])
    draw_params = post.isel(chain=chain, draw=draw)
    scalar_params = {p: float(v) for p, v in draw_params.items() if v.ndim == 0}
    multi_params = {p: v.values for p, v in draw_params.items() if v.ndim > 0 and p != "proc"}
    
    # Build the model
    pop = get_country_pop(iso3)
    data_start = find_run_start_time(pop, iso3)
    end_time = find_run_end_time(iso3, mob_source)
    run_start = data_start - timedelta(RUN_DATA_DELAY)
    start_str = run_start.strftime(DATE_FORMAT)
    end_str = data_start.strftime(DATE_FORMAT)
    var_data = get_country_vars(iso3)
    delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
    alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
    var_names = ["eu"] + alpha_var + delta_var  # Not applicable for Omicron-era countries
    mob_provider = get_mobility_provider(iso3, mob_source)
    model = MultiStrainModel(pop, run_start, end_time, var_names, alpha_seed + delta_seed, mob_provider, True, False)
    thinning = 7
    times = model.epoch.number_to_datetime(pd.Series(model.model_times))[::thinning]
    
    # Get synthetic indicators for calibration
    priors = get_standard_priors(len(var_names), "weekly_admissions", iso3, continent, False)
    prior_means = {}
    for k, v in priors.items():
        if v is None:
            continue
        elif isinstance(v, float):
            prior_means[k] = v
        else:
            prior_means[k] = v.mean
    proc = np.random.normal(0.0, 0.1, model.x_proc_vals.shape[0])
    mob_exp_param = {"mob_exp": random.uniform(EXP_PRIOR_LOWER, EXP_PRIOR_UPPER)}
    run_params = scalar_params | multi_params | prior_means | {"proc": proc} | mob_exp_param
    results = model.renewal_func(**run_params)
    
    # Calibrate
    mob_exp_dist = {"mob_exp": dist.Uniform(EXP_PRIOR_LOWER, EXP_PRIOR_UPPER)}
    calibrate_params = scalar_params | multi_params | prior_means | mob_exp_dist
    indicators = ["weekly_deaths", "weekly_cases"]
    outputs = {i: pd.Series(results[i][::thinning], index=times) for i in indicators}
    targets = {ind: SharedDispTarget(targ, weight=1.0 * targ.size) for ind, targ in outputs.items()}
    calib, mcmc = run_calibration(model, calibrate_params, targets, True, 10000)
    display(plot_input_recovery(priors, calib, az.from_numpyro(mcmc), outputs, mob_exp_param, proc))